In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import warnings


warnings.filterwarnings("ignore", category=UserWarning)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")    

class TumorFeatureDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.files = []
        self.labels = []
        
        class_mapping = {'normal': 0.0, 'tumor': 1.0}
        
        for class_name, label in class_mapping.items():
            class_dir = os.path.join(root_dir, class_name)
            if os.path.exists(class_dir):
                for f in os.listdir(class_dir):
                    if f.endswith('.pt'):
                        full_path = os.path.join(class_dir, f)
                        # Filter out blatantly empty files
                        if os.path.getsize(full_path) > 1024:
                            self.files.append(os.path.join(class_name, f))
                            self.labels.append(label)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = os.path.join(self.root_dir, self.files[idx])
        
        try:
            # Try to load the file
            features = torch.load(file_path).to(torch.float32)
            label = torch.tensor([self.labels[idx]], dtype=torch.float32)
            return features, label
            
        except Exception as e:
            # THE CATCH-ALL: Intercepts RuntimeErrors, EOFErrors, etc.
            print(f"\n⚠️ Skipping {self.files[idx]} | Reason: {type(e).__name__}")
            # Move to the next file
            return self.__getitem__((idx + 1) % len(self.files))

#  Initialize and Split Data
dataset_path = '/kaggle/input/datasets/vanshdistsys2/gigapixel-tumor-features/features'
full_dataset = TumorFeatureDataset(dataset_path)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f"Total Valid Slides: {len(full_dataset)} | Train: {train_size} | Val: {val_size}")

Training on: cuda
Total Valid Slides: 243 | Train: 194 | Val: 49


In [8]:


# 1. Clear the corrupted files from memory immediately
ignore_list = [
    'tumor/tumor_080.pt', 'tumor/tumor_092.pt', 'tumor/tumor_079.pt', 
    'tumor/tumor_078.pt', 'normal/normal_136.pt'
]

# Filtering the dataset list
full_dataset.files = [f for f in full_dataset.files if f not in ignore_list]

# Re-splitting the clean data
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

#  Define the  Architecture
class TumorClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TumorClassifier, self).__init__()
        
        # Layer 1: Expansion + Dropout for stability
        self.layer1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) 
        
        # Layer 2: Compression
        self.layer2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        
        # Layer 3 & Output
        self.layer3 = nn.Linear(256, 128)
        self.output = nn.Linear(128, 1)

    def forward(self, x):
        # x shape: (N, 2048)
        x = self.dropout(self.relu(self.bn1(self.layer1(x))))
        x = self.relu(self.bn2(self.layer2(x)))
        x = self.relu(self.layer3(x))
        
        
        patch_predictions = self.output(x) 
        
        
        slide_prediction = torch.max(patch_predictions, dim=0, keepdim=True)[0]
        
        return slide_prediction


model = TumorClassifier(2048).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)

print(f"Clean slides: {len(full_dataset)} | Ready for Max-Pooling run at LR=0.0001")

Clean slides: 238 | Ready for Max-Pooling run at LR=0.0001


In [9]:

epochs = 10

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train() 
    total_train_loss = 0
    
    for batch_features, batch_labels in train_loader:
        # Features come in as (1, N, 2048) -> squeeze(0) makes it (N, 2048)
        batch_features = batch_features.to(device).squeeze(0) 
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_features)
        loss = criterion(predictions, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval() 
    total_val_loss = 0
    correct_predictions = 0
    
    with torch.no_grad(): 
        for val_features, val_labels in val_loader:
            val_features = val_features.to(device).squeeze(0)
            val_labels = val_labels.to(device)
            
            val_preds = model(val_features)
            val_loss = criterion(val_preds, val_labels)
            total_val_loss += val_loss.item()
            
            # Accuracy calculation
            probabilities = torch.sigmoid(val_preds)
            predicted_classes = (probabilities > 0.5).float()
            correct_predictions += (predicted_classes == val_labels).sum().item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = (correct_predictions / len(val_loader)) * 100

    # SCREEN LOGGING (Clean - No Warnings!)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

Epoch 1/10 | Train Loss: 0.6903 | Val Loss: 0.6927 | Val Acc: 56.25%
Epoch 2/10 | Train Loss: 0.6818 | Val Loss: 0.7158 | Val Acc: 50.00%
Epoch 3/10 | Train Loss: 0.6813 | Val Loss: 0.7198 | Val Acc: 50.00%
Epoch 4/10 | Train Loss: 0.6809 | Val Loss: 0.7051 | Val Acc: 50.00%
Epoch 5/10 | Train Loss: 0.6748 | Val Loss: 0.6756 | Val Acc: 50.00%
Epoch 6/10 | Train Loss: 0.6739 | Val Loss: 0.6613 | Val Acc: 52.08%
Epoch 7/10 | Train Loss: 0.6731 | Val Loss: 0.7419 | Val Acc: 50.00%
Epoch 8/10 | Train Loss: 0.6525 | Val Loss: 0.6420 | Val Acc: 68.75%
Epoch 9/10 | Train Loss: 0.6402 | Val Loss: 0.6525 | Val Acc: 60.42%
Epoch 10/10 | Train Loss: 0.5910 | Val Loss: 0.5553 | Val Acc: 66.67%


Next Steps: Implementing Attention-Based MIL (ABMIL) to overcome the brittle nature of Max Pooling.